In [10]:
import pandas as pd
import pickle
import json
from collections import Counter
import ast
import networkx as nx
from IPython.display import display

In [2]:
def string_to_list_conversion(str_list):
    """convert the string representation of the list to an actual list
    """
    return ast.literal_eval(str_list)

def load_pickle(file):
    """
    load a pickle file into a local variable
    can be a pandas dataframe or dictionary
    """
    pickle_in = open(file, "rb")
    pickle_file = pickle.load(pickle_in)
    pickle_in.close()
    return pickle_file

In [3]:
df = pd.read_csv('pass_paths.csv')
df['path'] = df['path'].apply(string_to_list_conversion)
node_dict = load_pickle('../node_dictionary.pickle')
# Use the map function to create a new 'layer_source' column in the dataframe
df['layer_source'] = df['source'].map({obj: color for color, obj_list in node_dict.items() for obj in obj_list})
df

,path,source,target,layer_source
0,"[APOE_A1, MH3HEAD, MMSE]",APOE_A1,MMSE,GENETIC
1,"[APOE_A1, MH17MALI, MMSE]",APOE_A1,MMSE,GENETIC
2,"[APOE_A1, MH5RESP, MMSE]",APOE_A1,MMSE,GENETIC
3,"[APOE_A1, CEREB_TCC, MMSE]",APOE_A1,MMSE,GENETIC
4,"[APOE_A1, TOTAL_CSF, MMSE]",APOE_A1,MMSE,GENETIC
...,...,...,...,...
18337,"[MH18SURG, MH10GAST, UW_EF]",MH18SURG,UW_EF,RISKFACTORS
18338,"[MH18SURG, MH7DERM, UW_EF]",MH18SURG,UW_EF,RISKFACTORS
18339,"[MH18SURG, MH16SMOK, UW_EF]",MH18SURG,UW_EF,RISKFACTORS
18340,"[MH18SURG, MH12RENA, UW_EF]",MH18SURG,UW_EF,RISKFACTORS


In [4]:
df['layer_source'].value_counts()

MRI            7717
PET            4860
RISKFACTORS    3735
MOLECULAR      1473
GENETIC         557
Name: layer_source, dtype: int64

In [5]:
# Veces que aparecen los nodos en los paths
nodos_unicos = df['path'].explode().unique()
count_nodo_por_layer_source = {}
for layer, grupo in df.groupby('layer_source'):
    count_nodo = {}
    for fila in grupo['path']:
        for nodo in fila:
            if nodo in count_nodo:
                count_nodo[nodo] += 1
            else:
                count_nodo[nodo] = 1
    count_nodo_por_layer_source[layer] = count_nodo

dataframes_por_layer = []
for layer, count_nodo in count_nodo_por_layer_source.items():
    df_nodos = pd.DataFrame({'nodos': list(count_nodo.keys()), 'count_nodo': list(count_nodo.values())})
    df_nodos['layer_source'] = layer
    dataframes_por_layer.append(df_nodos)

df_resultado = pd.concat(dataframes_por_layer)
df_resultado['layer'] = df_resultado['nodos'].map({obj: color for color, obj_list in node_dict.items() for obj in obj_list})
df_resultado.sort_values(by=['count_nodo'], ascending=False, inplace=True)
df_resultado

,nodos,count_nodo,layer_source,layer
7,MH5RESP,1124,MRI,RISKFACTORS
4,CEREB_TCC,875,MRI,MRI
3,TOTAL_CSF,850,MRI,MRI
5,MH3HEAD,810,MRI,RISKFACTORS
8,MOCA,753,MRI,PHENOTYPE
...,...,...,...,...
86,ST91TA,1,MOLECULAR,MRI
88,L_HIPPO,1,MOLECULAR,MRI
172,AXCHEST,1,MRI,RISKFACTORS
137,AXMUSCLE,1,MRI,RISKFACTORS


In [6]:
# df_resultado[(df_resultado.layer_source == 'GENETIC') & (df_resultado.layer == 'PHENOTYPE')].head(5)

In this solution, we first create a list pairs that contains all pairs of consecutive nodes in the path column of the original dataframe. We do this by iterating over each row of the dataframe and over each element in the path list, and using the tuple function to create a pair of nodes.

We then use the Counter function from the Python standard library to count the number of times each pair appears in the pairs list.

Finally, we create a new dataframe df_pairs from the counts, by creating two columns: 'pair' with the pairs of nodes, and 'count' with the counts.

In [7]:
# create a list of all pairs of nodes in the paths
pairs = [tuple(df.loc[i, 'path'][j:j+2]) for i in range(len(df)) for j in range(len(df.loc[i, 'path'])-1)]
# count the pairs using Counter
pair_counts = Counter(pairs)

# create a new dataframe from the counts
df_pairs = pd.DataFrame({'pair': list(pair_counts.keys()), 'count': list(pair_counts.values())})
df_pairs['source'] = list(zip(*df_pairs['pair']))[0]
df_pairs['target'] = list(zip(*df_pairs['pair']))[1]
df_pairs

,pair,count,source,target
0,"(APOE_A1, MH3HEAD)",12,APOE_A1,MH3HEAD
1,"(MH3HEAD, MMSE)",128,MH3HEAD,MMSE
2,"(APOE_A1, MH17MALI)",12,APOE_A1,MH17MALI
3,"(MH17MALI, MMSE)",78,MH17MALI,MMSE
4,"(APOE_A1, MH5RESP)",12,APOE_A1,MH5RESP
...,...,...,...,...
4128,"(MH18SURG, MH12RENA)",12,MH18SURG,MH12RENA
4129,"(MH18SURG, MH13ALLE)",12,MH18SURG,MH13ALLE
4130,"(MH18SURG, MH10GAST)",8,MH18SURG,MH10GAST
4131,"(MH18SURG, MH7DERM)",10,MH18SURG,MH7DERM


In [8]:
# Crear el diccionario de path y pares
path_pairs = {}
for i in range(len(df)):
    pairs = [tuple(df.loc[i, 'path'][j:j+2]) for j in range(len(df.loc[i, 'path'])-1)]
    path_pairs[tuple(df.loc[i, 'path'])] = pairs

# # Mostrar el diccionario de pares
# for path, pairs in path_pairs.items():
#     print(f"Path: {path}")
#     print(f"Pares: {pairs}")
#     print()

new_df = pd.DataFrame({'path': key, 'pairs': value} for key, value in path_pairs.items())
new_df['sum_count'] = new_df['pairs'].apply(lambda pairs: df_pairs[df_pairs['pair'].isin(pairs)]['count'].sum())
new_df

,path,pairs,sum_count
0,"(APOE_A1, MH3HEAD, MMSE)","[(APOE_A1, MH3HEAD), (MH3HEAD, MMSE)]",140
1,"(APOE_A1, MH17MALI, MMSE)","[(APOE_A1, MH17MALI), (MH17MALI, MMSE)]",90
2,"(APOE_A1, MH5RESP, MMSE)","[(APOE_A1, MH5RESP), (MH5RESP, MMSE)]",187
3,"(APOE_A1, CEREB_TCC, MMSE)","[(APOE_A1, CEREB_TCC), (CEREB_TCC, MMSE)]",155
4,"(APOE_A1, TOTAL_CSF, MMSE)","[(APOE_A1, TOTAL_CSF), (TOTAL_CSF, MMSE)]",156
...,...,...,...
18337,"(MH18SURG, MH10GAST, UW_EF)","[(MH18SURG, MH10GAST), (MH10GAST, UW_EF)]",185
18338,"(MH18SURG, MH7DERM, UW_EF)","[(MH18SURG, MH7DERM), (MH7DERM, UW_EF)]",150
18339,"(MH18SURG, MH16SMOK, UW_EF)","[(MH18SURG, MH16SMOK), (MH16SMOK, UW_EF)]",54
18340,"(MH18SURG, MH12RENA, UW_EF)","[(MH18SURG, MH12RENA), (MH12RENA, UW_EF)]",43


Lo que queremos es separar según de qué capa es el source del path

In [11]:
layers = list(node_dict.keys())
del layers[-1]
all_nodes = []
for values in node_dict.values():
    all_nodes.extend(values)

for layer in layers:
    print(layer)
    df_temp = df[df.layer_source == layer].copy().reset_index(drop=True)

    pairs = [tuple(df_temp.loc[i, 'path'][j:j+2]) for i in range(len(df_temp)) for j in range(len(df_temp.loc[i, 'path'])-1)]
    pair_counts = Counter(pairs)

    df_pairs = pd.DataFrame({'pair': list(pair_counts.keys()), 'count': list(pair_counts.values())})
    df_pairs['source'] = list(zip(*df_pairs['pair']))[0]
    df_pairs['target'] = list(zip(*df_pairs['pair']))[1]

    path_pairs = {}
    for i in range(len(df)):
        pairs = [tuple(df.loc[i, 'path'][j:j+2]) for j in range(len(df.loc[i, 'path'])-1)]
        path_pairs[tuple(df.loc[i, 'path'])] = pairs
    new_df = pd.DataFrame({'path': key, 'pairs': value} for key, value in path_pairs.items())
    new_df['sum_count'] = new_df['pairs'].apply(lambda pairs: df_pairs[df_pairs['pair'].isin(pairs)]['count'].sum())
    new_df.sort_values(by=['sum_count'], ascending=False, inplace=True)
    display(new_df.head(10))

    G = nx.from_pandas_edgelist(df_pairs, source='source', target='target', edge_attr='count')
    G.add_nodes_from(all_nodes, ignore_existing=True) # add the rest of nodes without edges
    # G = nx.relabel_nodes(G, mapping)
    values = dict(G.degree)
    layers_net = {}
    for node in G:
        for layer_name in list(node_dict.keys()):
            if node in node_dict[layer_name]:
                layers_net[node] = layer_name
    nx.set_node_attributes(G, values, name='node_degree')
    nx.set_node_attributes(G, layers_net, name='layer')
    G_json = nx.readwrite.json_graph.cytoscape.cytoscape_data(G)
    # json.dump(G_json, open(f'path_net_{layer}.json', 'w'))

GENETIC


,path,pairs,sum_count
0,"(APOE_A1, MH3HEAD, MMSE)","[(APOE_A1, MH3HEAD), (MH3HEAD, MMSE)]",19
366,"(TOMM40_A2, MH3HEAD, ADSP_LAN)","[(TOMM40_A2, MH3HEAD), (MH3HEAD, ADSP_LAN)]",19
376,"(TOMM40_A2, MH5RESP, ADAS11)","[(TOMM40_A2, MH5RESP), (MH5RESP, ADAS11)]",19
375,"(TOMM40_A2, MH3HEAD, ADAS11)","[(TOMM40_A2, MH3HEAD), (MH3HEAD, ADAS11)]",19
131,"(APOE_A2, MH3HEAD, ADSP_LAN)","[(APOE_A2, MH3HEAD), (MH3HEAD, ADSP_LAN)]",19
133,"(APOE_A2, MH5RESP, ADSP_LAN)","[(APOE_A2, MH5RESP), (MH5RESP, ADSP_LAN)]",19
371,"(TOMM40_A2, MH5RESP, ADSP_VSP)","[(TOMM40_A2, MH5RESP), (MH5RESP, ADSP_VSP)]",19
370,"(TOMM40_A2, MH3HEAD, ADSP_VSP)","[(TOMM40_A2, MH3HEAD), (MH3HEAD, ADSP_VSP)]",19
138,"(APOE_A2, MH3HEAD, ADSP_VSP)","[(APOE_A2, MH3HEAD), (MH3HEAD, ADSP_VSP)]",19
139,"(APOE_A2, MH5RESP, ADSP_VSP)","[(APOE_A2, MH5RESP), (MH5RESP, ADSP_VSP)]",19


MOLECULAR


,path,pairs,sum_count
1086,"(TS_RATIO, TL, ST118TS, ADSP_VSP)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, ADSP...",126
1046,"(TS_RATIO, TL, ST118TS, ADSP_DX)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, ADSP...",126
1109,"(TS_RATIO, TL, ST118TS, ADAS13)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, ADAS...",125
1069,"(TS_RATIO, TL, ST118TS, ADSP_EXF)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, ADSP...",125
1121,"(TS_RATIO, TL, ST118TS, UW_MEM)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, UW_M...",125
1041,"(TS_RATIO, TL, ST118TS, CDR)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, CDR)]",125
1019,"(TS_RATIO, TL, ST118TS, MMSE)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, MMSE)]",125
1099,"(TS_RATIO, TL, ST118TS, ADAS11)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, ADAS...",125
1080,"(TS_RATIO, TL, ST118TS, ADSP_LAN)","[(TS_RATIO, TL), (TL, ST118TS), (ST118TS, ADSP...",124
1451,"(BACE, OCCMIDL02_FDG, ADAS13, UW_EF)","[(BACE, OCCMIDL02_FDG), (OCCMIDL02_FDG, ADAS13...",118


PET


,path,pairs,sum_count
3529,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, ADSP_MEM)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",132
3493,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, MMSE)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",132
3592,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, UW_EF)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",131
3565,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, ADAS11)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",131
3556,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, ADSP_VSP)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",130
3502,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, MOCA)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",130
3547,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, ADSP_LAN)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",130
3538,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, ADSP_EXF)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",130
3583,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, UW_MEM)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",130
3574,"(SUPMRGL01_FDG, HMEMOTIO, EUR_AB42/40, ADAS13)","[(SUPMRGL01_FDG, HMEMOTIO), (HMEMOTIO, EUR_AB4...",129


MRI


,path,pairs,sum_count
7748,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN, MMSE)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)...",202
7796,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN, ADSP_EXF)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)...",201
7787,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN, ADSP_MEM)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)...",201
7825,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN, ADAS11)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)...",201
7759,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN, MOCA)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)...",201
7073,"(CEREB_TCC, TOTAL_CSF, ADSP_EXF, MMSE)","[(CEREB_TCC, TOTAL_CSF), (TOTAL_CSF, ADSP_EXF)...",200
7843,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN, UW_MEM)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)...",200
7852,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN, UW_EF)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)...",200
7130,"(CEREB_TCC, TOTAL_CSF, ADSP_EXF, ADSP_LAN)","[(CEREB_TCC, TOTAL_CSF), (TOTAL_CSF, ADSP_EXF)...",199
7798,"(TOTAL_CSF, CEREB_TCC, ADSP_LAN)","[(TOTAL_CSF, CEREB_TCC), (CEREB_TCC, ADSP_LAN)]",199


RISKFACTORS


,path,pairs,sum_count
15738,"(AXRASH, MH7DERM, MH3HEAD, ADSP_DX)","[(AXRASH, MH7DERM), (MH7DERM, MH3HEAD), (MH3HE...",203
15773,"(AXRASH, MH7DERM, MH3HEAD, ADSP_VSP)","[(AXRASH, MH7DERM), (MH7DERM, MH3HEAD), (MH3HE...",198
15734,"(AXRASH, MH7DERM, MH3HEAD, CDR)","[(AXRASH, MH7DERM), (MH7DERM, MH3HEAD), (MH3HE...",197
15758,"(AXRASH, MH7DERM, MH3HEAD, ADSP_EXF)","[(AXRASH, MH7DERM), (MH7DERM, MH3HEAD), (MH3HE...",189
15781,"(AXRASH, MH7DERM, MH3HEAD, ADAS11)","[(AXRASH, MH7DERM), (MH7DERM, MH3HEAD), (MH3HE...",187
15751,"(AXRASH, MH7DERM, MH3HEAD, ADSP_MEM)","[(AXRASH, MH7DERM), (MH7DERM, MH3HEAD), (MH3HE...",186
15788,"(AXRASH, MH7DERM, MH3HEAD, ADAS13)","[(AXRASH, MH7DERM), (MH7DERM, MH3HEAD), (MH3HE...",185
16880,"(MH2NEURL, MH7DERM, MH3HEAD, ADSP_DX)","[(MH2NEURL, MH7DERM), (MH7DERM, MH3HEAD), (MH3...",182
16913,"(MH2NEURL, MH7DERM, MH3HEAD, ADSP_VSP)","[(MH2NEURL, MH7DERM), (MH7DERM, MH3HEAD), (MH3...",177
16946,"(MH2NEURL, MH7DERM, MH3HEAD, UW_EF)","[(MH2NEURL, MH7DERM), (MH7DERM, MH3HEAD), (MH3...",171
